In [1]:
# creating spark session
exec(open('/home/jovyan/.ipython/profile_default/startup/00-spark-session.py').read())

Spark 3.5.0 session ready as `spark` (Delta Lake enabled).


# Game Play First Login

## Problem Description

You are given a PySpark DataFrame named **`activity`** containing player activity records.

Each row represents a player's activity on a particular date.

Your task is to find the **first login date for each player**.

The first login is the earliest `event_date` recorded for that player.

### Input DataFrame

The `activity` DataFrame contains:

| Column | Data Type | Description |
|---|---|---|
| `player_id` | Integer | Unique identifier of the player |
| `device_id` | Integer | Identifier of the device used |
| `event_date` | String | Date of the activity in `YYYY-MM-DD` format |
| `games_played` | Integer | Number of games played on that date |

### Expected Output

Return the following columns:

- `player_id`
- `first_login`

The result should be sorted by `player_id` in ascending order.

### Example

| player_id | device_id | event_date | games_played |
|---:|---:|---|---:|
| 1 | 2 | 2023-01-01 | 5 |
| 1 | 2 | 2023-01-05 | 6 |
| 2 | 3 | 2023-01-02 | 1 |
| 3 | 1 | 2023-01-03 | 0 |
| 3 | 4 | 2023-01-01 | 2 |

### Expected Result

| player_id | first_login |
|---:|---|
| 1 | 2023-01-01 |
| 2 | 2023-01-02 |
| 3 | 2023-01-01 |

### Problem Pattern

**GROUP BY → MIN → RENAME → ORDER BY**

Use the minimum `event_date` for each `player_id` to determine the first login date.

In [2]:
data = [
    (1, 2, "2023-01-01", 5),
    (1, 2, "2023-01-05", 6),
    (2, 3, "2023-01-02", 1),
    (3, 1, "2023-01-03", 0),
    (3, 4, "2023-01-01", 2),
    (4, 2, "2023-01-07", 3),
    (4, 5, "2023-01-03", 4),
    (5, 1, "2023-01-10", 2),
    (5, 1, "2023-01-12", 5),
    (6, 3, "2023-02-01", 1),
    (6, 4, "2023-01-25", 7),
    (7, 2, "2023-02-10", 3),
    (7, 2, "2023-02-05", 6)
]

columns = [
    "player_id",
    "device_id",
    "event_date",
    "games_played"
]

activity = spark.createDataFrame(data, columns)

activity.show()

+---------+---------+----------+------------+
|player_id|device_id|event_date|games_played|
+---------+---------+----------+------------+
|        1|        2|2023-01-01|           5|
|        1|        2|2023-01-05|           6|
|        2|        3|2023-01-02|           1|
|        3|        1|2023-01-03|           0|
|        3|        4|2023-01-01|           2|
|        4|        2|2023-01-07|           3|
|        4|        5|2023-01-03|           4|
|        5|        1|2023-01-10|           2|
|        5|        1|2023-01-12|           5|
|        6|        3|2023-02-01|           1|
|        6|        4|2023-01-25|           7|
|        7|        2|2023-02-10|           3|
|        7|        2|2023-02-05|           6|
+---------+---------+----------+------------+



# Using SparkSQL

In [3]:
activity.createOrReplaceTempView("activity")

In [16]:
spark.sql("""
    SELECT
        player_id,
        MIN(CAST(event_date AS DATE)) AS first_login
    FROM activity
    GROUP BY player_id
    ORDER BY player_id
""").show()

+---------+-----------+
|player_id|first_login|
+---------+-----------+
|        1| 2023-01-01|
|        2| 2023-01-02|
|        3| 2023-01-01|
|        4| 2023-01-03|
|        5| 2023-01-10|
|        6| 2023-01-25|
|        7| 2023-02-05|
+---------+-----------+



# Using Pyspark

In [22]:
from pyspark.sql.functions import *

result = (
    activity
    .groupBy("player_id")
    .agg(
        min(col("event_date").cast("date")).alias("first_login")
    )
    .orderBy(col("player_id").asc())
)

result.show()

+---------+-----------+
|player_id|first_login|
+---------+-----------+
|        1| 2023-01-01|
|        2| 2023-01-02|
|        3| 2023-01-01|
|        4| 2023-01-03|
|        5| 2023-01-10|
|        6| 2023-01-25|
|        7| 2023-02-05|
+---------+-----------+

